# Extraction of analytical techniques from PDFs

This notebook scans all PDF files in a given folder and detects which analytical
techniques are mentioned in each paper, based on a predefined list of keywords.

**Input:**
- Folder with PDFs (e.g. `pdfs/`) generated by the DOI → PDF pipeline.

**Output:**
- A CSV file `techniques_by_paper.csv` with, for each PDF:
  - `pdf_file`
  - `doi` (if inferred from file name)
  - `techniques` (list of detected techniques)
  - `matched_keywords` (the actual keywords found in the text)

You only need to:
1. Adjust the configuration in the next cell (paths).
2. Run the notebook from top to bottom.

## Configuración

In [9]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Set

import pdfplumber  # pip install pdfplumber
import csv

# -------------------------
# Configuration
# -------------------------

# Folder containing the PDFs previously downloaded
PDF_FOLDER = Path("pdfs")

# Output CSV file
OUTPUT_CSV = Path("techniques_by_paper.csv")

# Whether to try to infer DOI from the PDF file name
# (Assumes files were saved as doi.replace('/', '_') + ".pdf")
INFER_DOI_FROM_FILENAME = True


# Safety check: ensure PDF folder exists
if not PDF_FOLDER.exists():
    raise FileNotFoundError(f"PDF folder not found: {PDF_FOLDER.resolve()}")

print(f"PDF folder: {PDF_FOLDER.resolve()}")
print(f"Output CSV: {OUTPUT_CSV.resolve()}")


PDF folder: /Users/pamelabenavides/repos/doi_pipeline/pdfs
Output CSV: /Users/pamelabenavides/repos/doi_pipeline/techniques_by_paper.csv


In [ ]:
# -------------------------
# Dictionary of techniques and associated keywords
# -------------------------

TECHNIQUES = {
    # 1. Molecular Biology Techniques
    "western_blot": [
        "western blot", "immunoblot", "wb analysis", "western"
    ],
    "rt_qpcr": [
        "rt-qpcr", "rt qpcr", "reverse transcription quantitative pcr",
        "real-time pcr", "real time pcr", "qpcr", "q-pcr"
    ],
    "ddpcr": [
        "digital droplet pcr", "droplet digital pcr", "ddpcr"
    ],
    "conventional_pcr": [
        "conventional pcr", "endpoint pcr", "pcr amplification"
    ],
    "elisa": [
        "elisa", "enzyme-linked immunosorbent assay"
    ],
    "chip_qpcr": [
        "chip-qpcr", "chromatin immunoprecipitation qpcr"
    ],
    "cell_transfection": [
        "cell transfection", "transfection"
    ],
    "sirna_knockdown": [
        "sirna knockdown", "sirna transfection", "gene silencing"
    ],

    # 2. Sequencing Technologies
    "illumina_rna_seq": [
        "rna-seq", "rna seq", "transcriptome sequencing",
        "illumina sequencing", "ngs rna-seq", "rna sequencing"
    ],
    "small_rna_seq": [
        "small rna sequencing", "small rna-seq", "mirna sequencing"
    ],
    "targeted_panel_ngs": [
        "targeted ngs", "panel-based sequencing", "amplicon sequencing",
        "targeted sequencing"
    ],
    "whole_exome_sequencing": [
        "whole exome sequencing", "wes"
    ],
    "whole_genome_sequencing": [
        "whole genome sequencing", "wgs"
    ],
    "single_molecule_sequencing": [
        "single-molecule sequencing", "single molecule sequencing",
        "smrt sequencing", "pacbio", "pacbio sequencing",
        "oxford nanopore", "nanopore sequencing", "ont sequencing",
        "minion", "promethion", "flongle"
    ],

    "sirna_knockdown": [
        "sirna knockdown",
        "sirna transfection",
        "gene silencing",
        "gene knockdown",
        "stable knockdown"
    ]

    # 3. Epigenetics / Methylation
    "bisulfite_sequencing": [
        "bisulfite sequencing", "bs-seq"
    ],
    "methylation_specific_pcr": [
        "methylation-specific pcr", "msp", "msp pcr"
    ],
    "cfDNA_methylation": [
        "cfdna methylation", "cell-free dna methylation"
    ],
    "methylation_mrd_assays": [
        "methylation-based mrd", "mrd methylation assay"
    ],

    # 4. Liquid Biopsy
    "liquid_biopsy_ngs": [
        "liquid biopsy ngs", "cfdna ngs", "circulating dna ngs"
    ],
    "ultra_deep_sequencing": [
        "ultra-deep sequencing", "deep sequencing"
    ],
    "cfdna_fragmentomics": [
        "fragmentomics", "dna fragmentation analysis"
    ],
    "ctdna_analysis": [
        "ctdna analysis", "circulating tumor dna analysis"
    ],

    # 5. Circulating Tumor Cells (CTCs)
    "ctc_cellsearch": [
        "cellsearch", "ctc enumeration", "circulating tumor cell detection"
    ],
    "ctc_qpcr": [
        "ctc qpcr", "qpcr ctc", "ctc-based qpcr"
    ],
    "neimfish": [
        "neimfish", "ne-imfish"
    ],

    # 6. Proteomics
    "mass_spectrometry": [
        "mass spectrometry", "lc-ms", "lc-ms/ms", "tims tof"
    ],
    "dia_proteomics": [
        "data-independent acquisition", "dia proteomics", "dia-ms"
    ],
    "tmt_proteomics": [
        "tmt proteomics", "tandem mass tag", "tmt-based quantification"
    ],

    # 7. Microscopy & Histology
    "immunohistochemistry": [
        "immunohistochemistry", "ihc staining", "ihc analysis", "ihc"
    ],
    "immunofluorescence": [
        "immunofluorescence", "if staining", "if microscopy"
    ],
    "h_e_staining": [
        "hematoxylin and eosin staining", "h&e staining", "h&e"
    ],
    "sirius_red_staining": [
        "sirius red", "sirius red staining"
    ],
    "trichrome_staining": [
        "masson trichrome", "trichrome staining"
    ],
    "confocal_microscopy": [
        "confocal microscopy", "confocal imaging"
    ],
    "electron_microscopy": [
        "electron microscopy", "tem", "transmission electron microscopy",
        "sem", "scanning electron microscopy"
    ],

    # 8. Flow Cytometry
    "flow_cytometry": [
        "flow cytometry", "facs analysis", "facs sorting"
    ],
    "cell_cycle_analysis": [
        "cell cycle analysis", "cell cycle assay"
    ],

    # 9. RNA-level Technologies
    "nanostring": [
        "nanostring", "nanostring ncounter", "n-counter analysis"
    ],
    "spatial_transcriptomics": [
        "spatial transcriptomics", "10x visium", "nanostring geomx"
    ],
    "mirna_qpcr": [
        "mirna qpcr", "microrna qpcr"
    ],
    "mirna_ddpcr": [
        "mirna ddpcr", "microrna ddpcr"
    ],

    # 10. Exosome-specific techniques
    "exosome_isolation": [
        "exosome isolation", "affinity membrane separation",
        "affinity adsorption", "exosome extraction"
    ],
    "particle_size_analysis": [
        "particle size analysis", "nta", "nanoparticle tracking analysis"
    ],
    "dynabeads_exosome_capture": [
        "dynabeads-based separation",
        "magnetic bead exosome capture",
        "antibody-coated magnetic beads",
        "dual-antibody fluorescent dynabeads",
        "immunomagnetic exosome isolation"
    ],

    # 11. Functional Cellular Assays
    "cell_viability_assay": [
        "cell viability assay", "viability assay"
    ],
    "scratch_assay": [
        "scratch assay", "wound healing assay"
    ],
    "transwell_assay": [
        "transwell assay", "migration assay", "invasion assay"
    ],
    "dual_luciferase_assay": [
        "dual luciferase reporter assay", "luciferase assay"
    ],
    "cck8_assay": [
        "cck8 assay", "cck-8", "cell counting kit-8"
    ],
    "edu_assay": [
        "edu assay", "edu incorporation", "dna synthesis assay"
    ],
    "colony_formation_assay": [
        "colony formation assay", "clonogenic assay"
    ],

    # 12. Bioinformatics (Non-statistical)
    "gsea_analysis": [
        "gsea", "gene set enrichment analysis"
    ],
    "cibersort_analysis": [
        "cibersort", "immune cell deconvolution"
    ],
    "go_kegg_enrichment": [
        "go analysis", "kegg analysis", "functional enrichment"
    ],
    "mirna_target_prediction": [
        "mirna target prediction", "targetscan", "mirdb", "mircode"
    ],
    "molecular_docking": [
        "molecular docking"
    ],

    # 13. AI / Multiomics
    "multiomics": [
        "multiomic profiling", "multi-omics", "proteogenomics",
        "integrated omics"
    ],
    "ai_ml_methods": [
        "machine learning", "deep learning", "ai-based analysis",
        "predictive modeling", "classification model"
    ],

    # 14. Viral / Expression Systems
    "lentiviral_transduction": [
        "lentiviral transduction", "shrna lentiviral vector",
        "viral transduction", "lentiviral infection"
    ],
    "plasmid_overexpression": [
        "plasmid overexpression", "gene overexpression",
        "transient overexpression"
    ],

    # 15. Chromatography / Biophysics
    "uhplc_ms": [
        "uhplc-qe-ms", "uhplc-ms", "uhplc-q exactive"
    ],
    "spr_assay": [
        "surface plasmon resonance", "spr assay"
    ],
    "cetsa_assay": [
        "cetsa", "cellular thermal shift assay"
    ],

    # 16. CTC-specific assays
    "csv_ctc_detection": [
        "csv-based ctc detection",
        "cell surface vimentin ctcs",
        "csv+ ctcs",
        "csv selection"
    ],

    # 17. m6A / RNA modification assays
    "merip_qpcr": [
        "merip-qpcr",
        "m6a merip",
        "rna immunoprecipitation qpcr"
    ],
    "select_qpcr": [
        "select assay",
        "single-base elongation and ligation-based qpcr",
        "select-qpcr"
    ],

    # 18. Tumor-informed / multiplex NGS
    "multiplex_pcr_ngs": [
        "multiplex pcr ngs",
        "personalized multiplex pcr",
        "tumor-informed multiplex assay",
        "personalized amplicon panel"
    ],
    #19. other techniques can be added here
    "dual_fluorescence_assay": [
        "dual fluorescence reporter assay",
        "dual-fluorescence assay"
    ],

    "fish_assay": [
        "fish",
        "fluorescence in situ hybridization"
    ],

    "apoptosis_assay": [
        "apoptosis assay",
        "annexin v",
        "tunnel assay",
        "caspase activity assay"
    ],

    "circrna_microarray": [
        "circrna microarray",
        "circrna array profiling"
    ],

    "rna_pull_down": [
        "rna pull-down",
        "biotin-labeled rna pull down"
    ],

    "rna_immunoprecipitation": [
        "rna immunoprecipitation",
        "rip assay"
    ],

    "primary_fibroblast_isolation": [
        "primary fibroblast isolation",
        "primary cell isolation",
        "caf isolation",
        "primary fibroblast culture",
        "isolation of cancer-associated fibroblasts"
    ],
    
    "exosome_marker_detection": [
        "exosome marker detection",
        "cd63 detection",
        "cd81 detection",
        "tsg101",
        "alix",
        "exosome surface markers"
    ],

    "targeted_bisulfite_sequencing": [
        "targeted bisulfite sequencing",
        "locus-specific bisulfite sequencing",
        "bisulfite sequencing of promoter",
        "promoter bisulfite sequencing"
    ]

}


print(f"Loaded {len(TECHNIQUES)} technique categories.")


Loaded 76 technique categories.


In [11]:
# -------------------------
# PDF text extraction helpers
# -------------------------

def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Extracts text from all pages of a PDF file.
    - First tries pdfplumber (if installed).
    - If pdfplumber is not available or fails, falls back to PyPDF2.
    - If both fail, returns an empty string.
    """
    texts: List[str] = []

    # Try pdfplumber first
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text() or ""
                texts.append(page_text)
        return "\n".join(texts)
    except Exception as e:
        print(f"  Warning: pdfplumber could not read {pdf_path.name}: {e}")
        print("  Trying PyPDF2 as fallback...")

    # Fallback: PyPDF2
    try:
        from PyPDF2 import PdfReader
        reader = PdfReader(str(pdf_path))
        for page in reader.pages:
            try:
                page_text = page.extract_text() or ""
            except Exception:
                page_text = ""
            texts.append(page_text)
        return "\n".join(texts)
    except Exception as e:
        print(f"  Warning: PyPDF2 also failed for {pdf_path.name}: {e}")
        print("  Skipping this file (no text extracted).")
        return ""



def maybe_extract_methods_section(full_text: str) -> str:
    """
    (Optional) Try to focus on the Methods section only.
    For simplicity and robustness, we keep it very basic:
    if a 'methods' heading is found, we keep text from there
    until the next major heading like 'results'.
    If nothing is found, we return the full text.
    """
    text_lower = full_text.lower()

    # Simple patterns for headings
    methods_idx = text_lower.find("methods")
    materials_methods_idx = text_lower.find("materials and methods")
    start_candidates = [i for i in [materials_methods_idx, methods_idx] if i != -1]

    if not start_candidates:
        return full_text  # fallback: use everything

    start = min(start_candidates)

    # Try to find an end heading
    end_keywords = ["results", "discussion", "conclusion"]
    end_positions = [
        text_lower.find(kw, start + 20) for kw in end_keywords
    ]
    end_positions = [pos for pos in end_positions if pos != -1]

    if end_positions:
        end = min(end_positions)
        return full_text[start:end]

    # If no end found, return from methods to end
    return full_text[start:]


def infer_doi_from_filename(pdf_path: Path) -> str:
    """
    Attempts to infer a DOI from the PDF file name.
    Assumes files were saved as doi.replace('/', '_') + '.pdf',
    e.g. '10.1000_j.journal.12345.pdf' → '10.1000/j.journal.12345'
    """
    name = pdf_path.name
    if not name.lower().endswith(".pdf"):
        return ""

    base = name[:-4]  # remove .pdf
    # Replace underscores back to slashes (best-effort)
    return base.replace("_", "/")


In [12]:
# -------------------------
# Technique detection
# -------------------------

def detect_techniques_in_text(
    text: str,
    techniques_dict: Dict[str, List[str]],
) -> Tuple[Set[str], Set[str]]:
    """
    Scans the text and detects which techniques appear.
    Returns:
      - set of technique keys detected
      - set of specific keyword matches
    """
    text_lower = text.lower()
    detected_techniques: Set[str] = set()
    matched_keywords: Set[str] = set()

    for tech_key, keywords in techniques_dict.items():
        for kw in keywords:
            kw_lower = kw.lower()
            if kw_lower in text_lower:
                detected_techniques.add(tech_key)
                matched_keywords.add(kw)
    return detected_techniques, matched_keywords


In [13]:
# -------------------------
# Main pipeline 
# -------------------------

def process_all_pdfs(
    pdf_folder: Path = PDF_FOLDER,
    output_csv: Path = OUTPUT_CSV,
    techniques_dict: Dict[str, List[str]] | None = None,
    infer_doi: bool = INFER_DOI_FROM_FILENAME,
) -> None:
    """
    Iterates over all PDFs in the given folder, extracts text,
    detects techniques and writes a summary CSV.

    This version is robust: any error in a single PDF is logged and
    the loop continues with the remaining files.
    """
    # Si no se pasa un diccionario, usar el global TECHNIQUES
    if techniques_dict is None:
        techniques_dict = TECHNIQUES

    print(f"Using {len(techniques_dict)} technique categories.")

    pdf_files = sorted(pdf_folder.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_folder}.")

    rows = []

    for pdf_path in pdf_files:
        print(f"\nProcessing: {pdf_path.name}")

        try:
            full_text = extract_text_from_pdf(pdf_path)

            if not full_text.strip():
                print("  Warning: no text extracted from this PDF.")
                detected = set()
                keywords = set()
            else:
                # Optionally restrict to methods section
                methods_text = maybe_extract_methods_section(full_text)
                detected, keywords = detect_techniques_in_text(
                    methods_text,
                    techniques_dict,
                )

            if infer_doi:
                doi = infer_doi_from_filename(pdf_path)
            else:
                doi = ""

            techniques_list = sorted(detected)
            keywords_list = sorted(keywords)

            rows.append(
                {
                    "pdf_file": pdf_path.name,
                    "doi": doi,
                    "techniques": "; ".join(techniques_list),
                    "matched_keywords": "; ".join(keywords_list),
                }
            )

            print(
                "  Techniques detected: "
                f"{', '.join(techniques_list) if techniques_list else 'none'}"
            )

        except Exception as e:
            print(f"  ERROR processing {pdf_path.name}: {e}")
            rows.append(
                {
                    "pdf_file": pdf_path.name,
                    "doi": infer_doi_from_filename(pdf_path) if infer_doi else "",
                    "techniques": "",
                    "matched_keywords": "",
                }
            )
            continue

    with output_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["pdf_file", "doi", "techniques", "matched_keywords"],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nDone. Summary saved to: {output_csv}")


In [14]:
process_all_pdfs(techniques_dict=TECHNIQUES)

Using 76 technique categories.
Found 86 PDF files in pdfs.

Processing: 10.1002_1878-0261.12911.pdf
  Techniques detected: apoptosis_assay, cck8_assay, cell_cycle_analysis, cell_transfection, electron_microscopy, fish_assay, flow_cytometry, immunohistochemistry, mirna_target_prediction, particle_size_analysis, rna_immunoprecipitation, rt_qpcr, transwell_assay, western_blot, whole_exome_sequencing

Processing: 10.1002_cam4.3755.pdf
  Techniques detected: methylation_specific_pcr, rt_qpcr

Processing: 10.1002_jcla.24520.pdf
  Techniques detected: mirna_target_prediction, transwell_assay

Processing: 10.1007_s10120-021-01267-5.pdf
  Techniques detected: illumina_rna_seq, rt_qpcr

Processing: 10.1007_s10120-022-01313-w.pdf
  Techniques detected: electron_microscopy

Processing: 10.1007_s10120-024-01556-9.pdf
  Techniques detected: ddpcr

Processing: 10.1007_s12094-024-03628-9.pdf
  Techniques detected: rt_qpcr

Processing: 10.1007_s13258-021-01086-z.pdf
  Techniques detected: go_kegg_enric

Cannot set gray non-stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value


  Techniques detected: conventional_pcr, ddpcr, electron_microscopy, exosome_marker_detection, fish_assay, immunohistochemistry, particle_size_analysis, rt_qpcr, western_blot, whole_exome_sequencing

Processing: 10.3748_wjg.v28.i6.653.pdf


Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value


  Techniques detected: electron_microscopy, illumina_rna_seq, small_rna_seq

Processing: 10.3892_etm.2021.10294.pdf
  Techniques detected: cell_transfection, electron_microscopy, fish_assay, particle_size_analysis, rna_immunoprecipitation, rt_qpcr, transwell_assay, western_blot, whole_exome_sequencing

Processing: 10.3892_mmr.2020.11577.pdf
  Techniques detected: apoptosis_assay, electron_microscopy, flow_cytometry, particle_size_analysis

Processing: 10.3892_mmr.2020.11698.pdf
  Techniques detected: cell_transfection, electron_microscopy, fish_assay, particle_size_analysis, rt_qpcr, scratch_assay

Processing: 10.3892_mmr.2021.12339.pdf
  Techniques detected: electron_microscopy, particle_size_analysis

Processing: 10.3892_mmr.2022.12703.pdf
  Techniques detected: cell_transfection, conventional_pcr, electron_microscopy, fish_assay, immunofluorescence, immunohistochemistry, rt_qpcr

Processing: 10.3892_ol.2020.12294.pdf
  Techniques detected: cell_transfection, electron_microscopy, fis

In [15]:
import pandas as pd

df = pd.read_csv("techniques_by_paper.csv")

# Separar la columna 'techniques' en listas
df["techniques_list"] = df["techniques"].fillna("").apply(
    lambda x: [t.strip() for t in x.split(";") if t.strip()]
)

# Contar frecuencia de cada técnica
from collections import Counter

counter = Counter()
for ts in df["techniques_list"]:
    counter.update(ts)

print("Technique counts:")
for tech, count in counter.most_common():
    print(f"{tech}: {count}")

Technique counts:
rt_qpcr: 48
electron_microscopy: 46
particle_size_analysis: 42
whole_exome_sequencing: 29
fish_assay: 26
western_blot: 25
cell_transfection: 21
transwell_assay: 16
immunohistochemistry: 14
illumina_rna_seq: 12
cck8_assay: 9
mirna_target_prediction: 9
ddpcr: 7
apoptosis_assay: 6
rna_immunoprecipitation: 6
scratch_assay: 6
flow_cytometry: 5
exosome_marker_detection: 4
go_kegg_enrichment: 3
immunofluorescence: 3
colony_formation_assay: 3
conventional_pcr: 3
cell_cycle_analysis: 2
methylation_specific_pcr: 2
gsea_analysis: 2
elisa: 2
small_rna_seq: 2
dual_luciferase_assay: 2
cfdna_fragmentomics: 2
cibersort_analysis: 2
bisulfite_sequencing: 1
targeted_bisulfite_sequencing: 1
exosome_isolation: 1
rna_pull_down: 1
cetsa_assay: 1
chip_qpcr: 1
uhplc_ms: 1
edu_assay: 1
circrna_microarray: 1
tmt_proteomics: 1
cfDNA_methylation: 1
ctdna_analysis: 1
multiomics: 1
targeted_panel_ngs: 1
whole_genome_sequencing: 1
cell_viability_assay: 1
dia_proteomics: 1


In [18]:
# Input files
INFO_CSV = "outputs/papers_info_semantic.xlsx"      # doi, title, abstract
PDF_STATUS_CSV = "pdf_status.csv"           # doi, pdf_local, pdf_path
TECH_CSV = "techniques_by_paper.csv"        # pdf_file, doi, techniques, matched_keywords

# Output file
MASTER_CSV = "papers_with_techniques.csv"

# ---------------------------
# Load data
# ---------------------------

df_info = pd.read_excel(INFO_CSV, engine="openpyxl")
df_pdf = pd.read_csv(PDF_STATUS_CSV)
df_tech = pd.read_csv(TECH_CSV)

# ---------------------------
# Clean DOI in tech file
# ---------------------------
# Some DOI strings in filenames may contain extra underscores etc.
df_tech["doi"] = df_tech["doi"].fillna("").str.strip()

# ---------------------------
# Merge step by DOI (left join to keep all 98 DOIs)
# ---------------------------
df_master = (
    df_info
    .merge(df_pdf[["doi", "pdf_local", "pdf_path"]], on="doi", how="left")
    .merge(df_tech[["doi", "techniques", "matched_keywords"]], on="doi", how="left")
)

# ---------------------------
# Save final CSV
# ---------------------------
df_master.to_csv(MASTER_CSV, index=False)

print(f"Master file created: {MASTER_CSV}")
print(f"Rows: {len(df_master)}")
df_master.head()


Master file created: papers_with_techniques.csv
Rows: 98


,doi,title,abstract,pdf,techniques_x,matched_keywords_x,pdf_local,pdf_path,techniques_y,matched_keywords_y
0,10.1080/15548627.2021.1901204,Long noncoding RNA (lncRNA) EIF3J-DT induces c...,Chemotherapy is currently the main treatment f...,sí,illumina_rna_seq; immunohistochemistry; rt_qpc...,ihc; qpcr; rna seq; wes; western blot,sí,pdfs/10.1080_15548627.2021.1901204.pdf,apoptosis_assay; cell_transfection; colony_for...,apoptosis assay; cell transfection; colony for...
1,10.3390/cancers14205105,The Role of ctDNA in Gastric Cancer,Circulating tumour DNA (ctDNA) has potential a...,sí,ctdna_analysis; ddpcr; immunohistochemistry; m...,ctdna analysis; ddpcr; droplet digital pcr; ih...,no,NaN,ctdna_analysis; ddpcr; electron_microscopy; fi...,ctdna analysis; ddpcr; droplet digital pcr; fi...
2,10.62347/BVFO4627,Bioinformatics- and quantitative proteomics-ba...,\n\nBackground: The emergence of immune resist...,sí,immunohistochemistry; rt_qpcr; western_blot; w...,immunohistochemistry; qpcr; rt-qpcr; wes; west...,sí,pdfs/10.62347_BVFO4627.pdf,cell_transfection; cell_viability_assay; dia_p...,cell transfection; cell viability assay; data-...
3,10.1016/j.clinbiochem.2024.110767,Circulating miRNA and circulating tumor DNA ap...,\n\nBackground: Gastric cancer (GC) is the thi...,sí,bisulfite_sequencing; rt_qpcr,bisulfite sequencing; qpcr; real-time pcr; rt-...,no,NaN,bisulfite_sequencing; electron_microscopy; rt_...,bisulfite sequencing; qpcr; real-time pcr; rt-...
4,10.1016/j.cca.2024.117773,Exosomal miRNAs from neutrophils act as accura...,\n\nBackground: Gastric cancer (GC) is the thi...,sí,NaN,NaN,no,NaN,ddpcr; electron_microscopy; illumina_rna_seq,ddpcr; droplet digital pcr; rna seq; rna seque...
